In [1]:
import os
import sys
import math
import shutil

module_path = os.path.abspath('Detection')
sys.path.insert(0, module_path)

from train import *
from model import SSD300

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
# device = 'cpu'
device

'cuda'

In [3]:
CLASSES = ['__background__', 'RBC', 'WBC', 'Platelets']
NUM_CLASSES = len(CLASSES)
model = SSD300(n_classes=NUM_CLASSES)
model

/home/superadmin/venv/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/superadmin/venv/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/superadmin/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 528M/528M [04:46<00:00, 1.93MB/s]



Loaded base model.



SSD300(
  (base): VGGBase(
    (conv1_1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv1_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv2_1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2_2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv3_1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv3_2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv3_3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv4_1): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv4_2): Conv2d(512, 512, kernel_size=(3, 3), 

In [4]:
# Initialize the optimizer, with twice the default learning rate for biases, as in the original Caffe repo
biases = list()
not_biases = list()
for param_name, param in model.named_parameters():
    if param.requires_grad:
        if param_name.endswith('.bias'):
            biases.append(param)
        else:
            not_biases.append(param)
optimizer = torch.optim.SGD(params=[{'params': biases, 'lr': 2 * lr}, {'params': not_biases}],
                            lr=lr, momentum=momentum, weight_decay=weight_decay)

model = model.to(device)
criterion = MultiBoxLoss(priors_cxcy=model.priors_cxcy).to(device)

/home/superadmin/venv/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


## Датасет

In [5]:
trainval_rate = 0.9
train_rate = 0.8

images_path = 'BCCD_Dataset/BCCD/JPEGImages'
annotations_path = 'BCCD_Dataset/BCCD/Annotations'
output_dir = 'BCCD_Dataset/BCCD/result'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

images_names = os.listdir(images_path)
images_list = []
for img in images_names:
    images_list.append(img.split('.')[0])
random.shuffle(images_list)

annotation_num = len(images_list)
trainval_num = int(math.ceil(trainval_rate * annotation_num))
train_num = int(math.ceil(trainval_num * train_rate))
trainval = images_list[0:trainval_num]
test = images_list[trainval_num:]
train = trainval[0:train_num]
val = trainval[train_num:trainval_num]
trainval = sorted(trainval)
train = sorted(train)
val = sorted(val)
test = sorted(test)

train, val, test

TRAIN_DIR = os.path.join(output_dir, 'train')
TEST_DIR = os.path.join(output_dir, 'test')
VALID_DIR = os.path.join(output_dir, 'valid')

for dir in (TRAIN_DIR, TEST_DIR, VALID_DIR):
    if not os.path.exists(dir):
        os.makedirs(dir)

for example in train:
    shutil.copyfile(os.path.join(images_path, f'{example}.jpg'), 
                    os.path.join(TRAIN_DIR, f'{example}.jpg'))
    shutil.copyfile(os.path.join(annotations_path, f'{example}.xml'), 
                    os.path.join(TRAIN_DIR, f'{example}.xml'))

for example in test:
    shutil.copyfile(os.path.join(images_path, f'{example}.jpg'), 
                    os.path.join(TEST_DIR, f'{example}.jpg'))
    shutil.copyfile(os.path.join(annotations_path, f'{example}.xml'), 
                    os.path.join(TEST_DIR, f'{example}.xml'))

for example in val:
    shutil.copyfile(os.path.join(images_path, f'{example}.jpg'), 
                    os.path.join(VALID_DIR, f'{example}.jpg'))
    shutil.copyfile(os.path.join(annotations_path, f'{example}.xml'), 
                    os.path.join(VALID_DIR, f'{example}.xml'))

In [7]:
def train(train_loader, model, criterion, optimizer, epoch):
    """
    One epoch's training.

    :param train_loader: DataLoader for training data
    :param model: model
    :param criterion: MultiBox loss
    :param optimizer: optimizer
    :param epoch: epoch number
    """
    model.train()  # training mode enables dropout

    batch_time = AverageMeter()  # forward prop. + back prop. time
    data_time = AverageMeter()  # data loading time
    losses = AverageMeter()  # loss

    start = time.time()

    # Batches
    for i, (images, boxes, labels, _) in enumerate(train_loader):
        data_time.update(time.time() - start)

        # Move to default device
        images = images.to(device)  # (batch_size (N), 3, 300, 300)
        boxes = [b.to(device) for b in boxes]
        labels = [l.to(device) for l in labels]

        # Forward prop.
        predicted_locs, predicted_scores = model(images)  # (N, 8732, 4), (N, 8732, n_classes)

        # Loss
        loss = criterion(predicted_locs, predicted_scores, boxes, labels)  # scalar

        # Backward prop.
        optimizer.zero_grad()
        loss.backward()

        # Clip gradients, if necessary
        if grad_clip is not None:
            clip_gradient(optimizer, grad_clip)

        # Update model
        optimizer.step()

        losses.update(loss.item(), images.size(0))
        batch_time.update(time.time() - start)

        start = time.time()

        # Print status
        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Batch Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data Time {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'.format(epoch, i, len(train_loader),
                                                                  batch_time=batch_time,
                                                                  data_time=data_time, loss=losses))
    del predicted_locs, predicted_scores, images, boxes, labels  # free some memory since their histories may be stored

In [12]:
from torch.utils.data import Dataset
from PIL import Image
import xml.etree.ElementTree as et

class PascalVOCDatasetAdapted(Dataset):
    """
    A PyTorch Dataset class to be used in a PyTorch DataLoader to create batches.
    """

    def __init__(self, data_folder, split, keep_difficult=False):

        self.split = split.upper()
        assert self.split in {'TRAIN', 'TEST'}
        self.width = 300
        self.height = 300
        self.folder = os.path.join(data_folder, split.lower())
        self.images = list(filter(lambda x: x.endswith('jpg'), os.listdir(self.folder)))
        self.annotations = list(filter(lambda x: x.endswith('xml'), os.listdir(self.folder)))
        
        assert len(self.images) == len(self.annotations)
        

    def __getitem__(self, i):
        # Read image
        image = Image.open(os.path.join(self.folder, self.images[i]), mode='r')
        image = image.convert('RGB')

        annot_file_path = os.path.join(self.folder, self.annotations[i])
        boxes = []
        labels = []
        difficulties = []
        tree = et.parse(annot_file_path)
        root = tree.getroot()

        image_width = image.width
        image_height = image.height

        for member in root.findall('object'):
            # Get label and map the `classes`.
            labels.append(CLASSES.index(member.find('name').text))
            difficulties.append(int(member.find('difficult').text))
            
            # Left corner x-coordinates.
            xmin = int(member.find('bndbox').find('xmin').text)
            # Right corner x-coordinates.
            xmax = int(member.find('bndbox').find('xmax').text)
            # Left corner y-coordinates.
            ymin = int(member.find('bndbox').find('ymin').text)
            # Right corner y-coordinates.
            ymax = int(member.find('bndbox').find('ymax').text)
            
            # Resize the bounding boxes according 
            # to resized image `width`, `height`.
            xmin_final = (xmin/image_width)*self.width
            xmax_final = (xmax/image_width)*self.width
            ymin_final = (ymin/image_height)*self.height
            ymax_final = (ymax/image_height)*self.height

            # Check that all coordinates are within the image.
            if xmax_final > self.width:
                xmax_final = self.width
            if ymax_final > self.height:
                ymax_final = self.height
            
            boxes.append([xmin_final, ymin_final, xmax_final, ymax_final])

        labels = torch.as_tensor(labels, dtype=torch.int64)
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        difficulties = torch.as_tensor(difficulties, dtype=torch.uint8)
        image, boxes, labels, difficulties = transform(image, boxes, labels, difficulties, split=self.split)
        return image, boxes, labels, difficulties

    def __len__(self):
        return len(self.images)

    def collate_fn(self, batch):
        """
        Since each image may have a different number of objects, we need a collate function (to be passed to the DataLoader).

        This describes how to combine these tensors of different sizes. We use lists.

        Note: this need not be defined in this Class, can be standalone.

        :param batch: an iterable of N sets from __getitem__()
        :return: a tensor of images, lists of varying-size tensors of bounding boxes, labels, and difficulties
        """

        images = list()
        boxes = list()
        labels = list()
        difficulties = list()

        for b in batch:
            images.append(b[0])
            boxes.append(b[1])
            labels.append(b[2])
            difficulties.append(b[3])

        images = torch.stack(images, dim=0)

        return images, boxes, labels, difficulties  # tensor (N, 3, 300, 300), 3 lists of N tensors each

# Learning parameters
checkpoint = None  # path to model checkpoint, None if none
batch_size = 8  # batch size
iterations = 120000  # number of iterations to train
workers = 4  # number of workers for loading data in the DataLoader
print_freq = 200  # print training status every __ batches
lr = 1e-3  # learning rate
decay_lr_at = [80000, 100000]  # decay learning rate after these many iterations
decay_lr_to = 0.1  # decay learning rate to this fraction of the existing learning rate
momentum = 0.9  # momentum
weight_decay = 5e-4  # weight decay
grad_clip = None  # clip if gradients are exploding, which may happen at larger batch sizes (sometimes at 32) - you will recognize it by a sorting error in the MuliBox loss calculation

train_dataset = PascalVOCDatasetAdapted(data_folder='BCCD_Dataset/BCCD/result', split='train')
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                               collate_fn=train_dataset.collate_fn, num_workers=workers,
                                               pin_memory=True)  # note that we're passing the collate function here

epochs = iterations // (len(train_dataset) // 32)
decay_lr_at = [it // (len(train_dataset) // 32) for it in decay_lr_at]
start_epoch = 0

# Epochs
for epoch in range(start_epoch, epochs):

    # Decay learning rate at particular epochs
    if epoch in decay_lr_at:
        adjust_learning_rate(optimizer, decay_lr_to)

    # One epoch's training
    train(train_loader=train_loader,
          model=model,
          criterion=criterion,
          optimizer=optimizer,
          epoch=epoch)

    # Save checkpoint
    save_checkpoint(epoch, model, optimizer)

Epoch: [0][0/33]	Batch Time 2.836 (2.836)	Data Time 0.312 (0.312)	Loss 16.2816 (16.2816)	
Epoch: [1][0/33]	Batch Time 0.464 (0.464)	Data Time 0.382 (0.382)	Loss inf (inf)	
Epoch: [2][0/33]	Batch Time 0.492 (0.492)	Data Time 0.412 (0.412)	Loss nan (nan)	
Epoch: [3][0/33]	Batch Time 0.508 (0.508)	Data Time 0.430 (0.430)	Loss nan (nan)	
Epoch: [4][0/33]	Batch Time 0.642 (0.642)	Data Time 0.560 (0.560)	Loss nan (nan)	
Epoch: [5][0/33]	Batch Time 0.703 (0.703)	Data Time 0.624 (0.624)	Loss nan (nan)	
Epoch: [6][0/33]	Batch Time 0.536 (0.536)	Data Time 0.457 (0.457)	Loss nan (nan)	
Epoch: [7][0/33]	Batch Time 0.525 (0.525)	Data Time 0.443 (0.443)	Loss nan (nan)	
Epoch: [8][0/33]	Batch Time 0.450 (0.450)	Data Time 0.370 (0.370)	Loss nan (nan)	
Epoch: [9][0/33]	Batch Time 0.615 (0.615)	Data Time 0.538 (0.538)	Loss nan (nan)	
Epoch: [10][0/33]	Batch Time 0.641 (0.641)	Data Time 0.563 (0.563)	Loss nan (nan)	
Epoch: [11][0/33]	Batch Time 0.523 (0.523)	Data Time 0.444 (0.444)	Loss nan (nan)	
Epoch:

Exception in thread Thread-58:
Traceback (most recent call last):
  File "/usr/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/home/superadmin/venv/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/home/superadmin/venv/lib/python3.8/site-packages/torch/utils/data/_utils/pin_memory.py", line 55, in _pin_memory_loop
    do_one_step()
  File "/home/superadmin/venv/lib/python3.8/site-packages/torch/utils/data/_utils/pin_memory.py", line 32, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.8/multiprocessing/queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "/home/superadmin/venv/lib/python3.8/site-packages/torch/multiprocessing/reductions.py", line 496, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/pytho

KeyboardInterrupt: 